In [1]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional
import logging

import numpy as np
import pandas as pd
from sqlalchemy.orm import Session

# konfiguracja logowania w notebooku
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dataset-notebook")

# ustawienie ścieżki do backendu, żeby móc importować moduły app.*
PROJECT_ROOT = Path("..").resolve()
BACKEND_PATH = PROJECT_ROOT / "backend"
if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

from app.database import SessionLocal
from app.models.user import User
from app.models.symptom_profile import SymptomProfile
from app.services.embedding_service import generate_embedding, generate_embeddings_batch
from app.services.matching_service import find_matching_group, add_user_to_group
from app.services.group_characteristics import update_group_characteristics
from app.routers.auth import _hash_password

DATA_PATH = PROJECT_ROOT / "data" / "test_data.json"
DRY_RUN = False

/home/mario/miniconda3/envs/zebrapoint/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from contextlib import contextmanager
from collections import Counter


@contextmanager
def get_db() -> Session:
    """Kontekst menedżer na sesję DB (tak jak w backendzie, ale dla notebooka)."""
    db = SessionLocal()
    try:
        yield db
        db.commit()
    except Exception:
        db.rollback()
        raise
    finally:
        db.close()


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    """Wczytuje testowy zbiór danych z JSON-a."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Plik JSON powinien zawierać listę obiektów")

    records: List[Dict[str, Any]] = []
    for idx, raw in enumerate(data, start=1):
        email = raw.get("email")
        password = raw.get("haslo")
        description = raw.get("opis")
        display_name = raw.get("display_name")

        if not email or not password or not description:
            raise ValueError(f"Rekord {idx} ma brakujące pola (email/haslo/opis)")

        # display_name jest wymagany zgodnie z planem; jeśli go nie ma,
        # można doraźnie użyć prefiksu z emaila.
        if not display_name:
            display_name = email.split("@")[0]

        description = str(description).strip()
        if len(description) < 100:
            raise ValueError(
                f"Opis w rekordzie {idx} jest za krótki ({len(description)} znaków). "
                "Potrzebne >= 100 znaków."
            )

        rec = {
            "email": str(email).strip(),
            "password": str(password),
            "display_name": str(display_name).strip(),
            "description": description,
        }
        if raw.get("age_range") is not None:
            rec["age_range"] = str(raw["age_range"]).strip()
        records.append(rec)

    logger.info("Wczytano %d rekordów z %s", len(records), path)
    return records


def create_user_from_record(db: Session, record: Dict[str, Any]) -> User:
    """Tworzy użytkownika na podstawie rekordu z JSON-a albo zwraca istniejącego."""
    email = record["email"]
    display_name = record["display_name"]
    password = record["password"]

    user = db.query(User).filter(User.email == email).first()
    if user:
        logger.info("Użytkownik %s już istnieje — pomijam tworzenie", email)
        return user

    user = User(
        email=email,
        password_hash=_hash_password(password),
        display_name=display_name,
    )
    if record.get("age_range") is not None:
        user.age_range = record.get("age_range")
    db.add(user)
    db.flush()
    logger.info("Utworzono użytkownika %s", email)
    return user


def create_or_update_symptom_profile(
    db: Session,
    user: User,
    description: str,
) -> SymptomProfile:
    """Tworzy lub aktualizuje profil objawów i dopasowuje grupę."""
    profile = (
        db.query(SymptomProfile)
        .filter(SymptomProfile.user_id == user.id)
        .first()
    )

    embedding = generate_embedding(description)
    match = find_matching_group(db, embedding, user.id)
    group_id = match["group_id"]

    if profile is None:
        profile = SymptomProfile(
            user_id=user.id,
            description=description,
            embedding=embedding,
            group_id=group_id,
            match_score=match["score"],
        )
        db.add(profile)
        logger.info("Utworzono profil objawów dla %s", user.email)
    else:
        profile.description = description
        profile.embedding = embedding
        profile.group_id = group_id
        profile.match_score = match["score"]
        logger.info("Zaktualizowano profil objawów dla %s", user.email)

    add_user_to_group(db, user.id, group_id)

    db.flush()
    return profile

In [3]:
# Test połączenia z bazą Supabase

with get_db() as db:
    try:
        user_count = db.query(User).count()
        print(f"Połączenie OK. Liczba użytkowników w bazie: {user_count}")
    except Exception as exc:
        print("Błąd połączenia z bazą:", exc)
        raise

2026-03-14 13:45:22,515 INFO sqlalchemy.engine.Engine select pg_catalog.version()


INFO:sqlalchemy.engine.Engine:select pg_catalog.version()


2026-03-14 13:45:22,516 INFO sqlalchemy.engine.Engine [raw sql] {}


INFO:sqlalchemy.engine.Engine:[raw sql] {}


2026-03-14 13:45:22,576 INFO sqlalchemy.engine.Engine select current_schema()


...

INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users) AS anon_1


2026-03-14 13:45:22,708 INFO sqlalchemy.engine.Engine [generated in 0.00210s] {}


INFO:sqlalchemy.engine.Engine:[generated in 0.00210s] {}


Połączenie OK. Liczba użytkowników w bazie: 0
2026-03-14 13:45:22,768 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


In [4]:
# Wczytanie danych z JSON i podgląd struktury

records = load_dataset(DATA_PATH)
print(f"Liczba rekordów w zbiorze: {len(records)}")

df_preview = pd.DataFrame(records)
df_preview.head()

INFO:dataset-notebook:Wczytano 27 rekordów z /home/mario/workspace/zebrapoint/data/test_data.json


Liczba rekordów w zbiorze: 27


,email,password,display_name,description
0,test0004@zebra.pl,test1234,Mama 4,Nasze dziecko urodziło się z bardzo niską masą...
1,test0005@zebra.pl,test1234,Mama 5,Od urodzenia zauważyliśmy u córki bardzo jasną...
2,test0006@zebra.pl,test1234,Mama 6,Syn od pierwszych miesięcy życia miał trudnośc...
3,test0007@zebra.pl,test1234,Mama 7,Nasze dziecko od małego bardzo mało mówiło. Te...
4,test0008@zebra.pl,test1234,Mama 8,Córka od niemowlęctwa miała częste napady drga...


In [5]:
# Utworzenie użytkowników w tabeli `users` na podstawie rekordów

created = 0
existing = 0

with get_db() as db:
    for rec in records:
        user_before = db.query(User).filter(User.email == rec["email"]).first()
        user = create_user_from_record(db, rec)
        if user_before is None:
            created += 1
        else:
            existing += 1

print(f"Nowo utworzonych użytkowników: {created}")
print(f"Użytkownicy już istniejący:   {existing}")

2026-03-14 13:45:29,412 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-14 13:45:29,418 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


INFO:sqlalchemy.engine.Engine:SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


2026-03-14 13:45:29,420 INFO sqlalchemy.engine.Engine [generated in 0.00211s] {'email_1': 'test0004@zebra.pl', 'param_1': 1}


...

2026-03-14 13:45:38,611 INFO sqlalchemy.engine.Engine [cached since 8.843s ago] {'id': UUID('361e046d-e170-4a8b-ae7f-d9770735f9df'), 'email': 'test0030@zebra.pl', 'password_hash': '$2b$12$R4no.x9bpLIJfKO9Pk2urejrnLMCUn6GfQFWEaG38xoRWV938DsDW', 'display_name': 'Mama 30', 'avatar_url': None, 'role': 'user', 'age_range': None, 'is_active': True}


INFO:sqlalchemy.engine.Engine:[cached since 8.843s ago] {'id': UUID('361e046d-e170-4a8b-ae7f-d9770735f9df'), 'email': 'test0030@zebra.pl', 'password_hash': '$2b$12$R4no.x9bpLIJfKO9Pk2urejrnLMCUn6GfQFWEaG38xoRWV938DsDW', 'display_name': 'Mama 30', 'avatar_url': None, 'role': 'user', 'age_range': None, 'is_active': True}
INFO:dataset-notebook:Utworzono użytkownika test0030@zebra.pl


2026-03-14 13:45:38,646 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Nowo utworzonych użytkowników: 27
Użytkownicy już istniejący:   0


In [6]:
# Utworzenie / aktualizacja profili objawów i dopasowanie do grup

profiles_created = 0
profiles_updated = 0

with get_db() as db:
    for rec in records:
        email = rec["email"]
        user = db.query(User).filter(User.email == email).first()
        if user is None:
            logger.warning("Brak użytkownika %s — pomijam profil objawów", email)
            continue

        existing_profile = (
            db.query(SymptomProfile)
            .filter(SymptomProfile.user_id == user.id)
            .first()
        )

        profile = create_or_update_symptom_profile(db, user, rec["description"])

        if existing_profile is None:
            profiles_created += 1
        else:
            profiles_updated += 1

    # Uzupełnienie charakterystyk grup (keywords, symptom_category, avg_match_score),
    # gdy Celery nie działa — update_group_characteristics robi commit wewnętrznie.
    group_ids = {row[0] for row in db.query(SymptomProfile.group_id).all() if row[0] is not None}
    for gid in group_ids:
        update_group_characteristics(db, str(gid))

print(f"Nowo utworzonych profili objawów: {profiles_created}")
print(f"Zaktualizowanych profili objawów: {profiles_updated}")

2026-03-14 13:46:07,954 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-14 13:46:07,956 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


INFO:sqlalchemy.engine.Engine:SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


2026-03-14 13:46:07,958 INFO sqlalchemy.engine.Engine [cached since 38.54s ago] {'email_1': 'test0004@zebra.pl', 'param_1': 1}


...

2026-03-14 13:46:26,697 INFO sqlalchemy.engine.Engine [cached since 4.104s ago] {'pk_1': UUID('48c37ef2-d744-4e8f-99b2-a2a6ea2484cf')}


INFO:sqlalchemy.engine.Engine:[cached since 4.104s ago] {'pk_1': UUID('48c37ef2-d744-4e8f-99b2-a2a6ea2484cf')}
INFO:app.services.group_characteristics:Charakterystyki grupy 48c37ef2-d744-4e8f-99b2-a2a6ea2484cf zaktualizowane: keywords=['zauważyliśmy', 'trudności', 'nasza', 'małego', 'spokojna'], category=Neurologiczne, age_range=None, avg_score=0.374


2026-03-14 13:46:26,754 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Nowo utworzonych profili objawów: 27
Zaktualizowanych profili objawów: 0


In [7]:
# Diagnostyka: liczba profili i rozkład po grupach

with get_db() as db:
    total_profiles = db.query(SymptomProfile).count()
    print(f"Liczba profili objawów w bazie: {total_profiles}")

    rows = (
        db.query(SymptomProfile.group_id)
        .all()
    )


group_ids = [row[0] for row in rows]
counts = Counter(group_ids)

print("\nRozkład liczby profili w grupach:")
for group_id, cnt in counts.items():
    print(f"  grupa={group_id}: {cnt} profili")

2026-03-14 13:47:22,449 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-14 13:47:22,455 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM (SELECT symptom_profiles.id AS symptom_profiles_id, symptom_profiles.user_id AS symptom_profiles_user_id, symptom_profiles.description AS symptom_profiles_description, symptom_profiles.embedding AS symptom_profiles_embedding, symptom_profiles.group_id AS symptom_profiles_group_id, symptom_profiles.match_score AS symptom_profiles_match_score, symptom_profiles.created_at AS symptom_profiles_created_at, symptom_profiles.updated_at AS symptom_profiles_updated_at 
FROM symptom_profiles) AS anon_1


INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT symptom_profiles.id AS symptom_profiles_id, symptom_profiles.user_id AS symptom_profiles_user_id, symptom_profiles.description AS symptom_profiles_description, symptom_profiles.embedding AS symptom_profiles_embedding, symptom_profiles.group_id AS symptom_profiles_group_id, symptom_profiles.match_score AS symptom_profiles_match_score, symptom_profiles.created_at AS symptom_profiles_created_at, symptom_profiles.updated_at AS symptom_profiles_updated_at 
FROM symptom_profiles) AS anon_1


2026-03-14 13:47:22,456 INFO sqlalchemy.engine.Engine [generated in 0.00142s] {}


...

2026-03-14 13:47:22,515 INFO sqlalchemy.engine.Engine [cached since 60.15s ago] {}


INFO:sqlalchemy.engine.Engine:[cached since 60.15s ago] {}


2026-03-14 13:47:22,545 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT



Rozkład liczby profili w grupach:
  grupa=118357ca-fb34-4ccf-9d33-f063823ad0a0: 2 profili
  grupa=8f35f4a1-0f5a-4625-bf76-d4201c60eabf: 2 profili
  grupa=8a64404b-26f3-4526-97ef-c661e5422394: 1 profili
  grupa=e3613a2c-cecb-4cf1-a5e0-e185e0f685f5: 1 profili
  grupa=eac982c3-c49e-4311-82ac-53a69a85772a: 2 profili
  grupa=ed2f7825-68df-4473-a77a-987d4bdbaab2: 1 profili
  grupa=d9a4ada7-fcc7-4deb-83e5-ea5f07fcbc71: 6 profili
  grupa=ef27b8bd-eb2a-410b-8cb8-d4565ef2d5b6: 1 profili
  grupa=e7146571-4b36-4e85-8af6-b1a7b8db9d6a: 1 profili
  grupa=48c37ef2-d744-4e8f-99b2-a2a6ea2484cf: 2 profili
  grupa=07b44788-66d9-4453-8a6d-947894852b7b: 1 profili
  grupa=3a1d6889-1ae0-4dbb-86f1-5dd94a0ef1fd: 2 profili
  grupa=c797f3f4-ebc0-4dce-aaad-e0d80c10ba5e: 1 profili
  grupa=3cf2e0ce-5055-4313-8fa9-9e2039918cbe: 1 profili
  grupa=3e1cde30-bbdc-47bb-a8b2-516f24488438: 1 profili
  grupa=0a816e83-f552-4924-a761-c6c1c5b82c8f: 1 profili
  grupa=2ce00ca2-8e38-43f1-afd1-5fafb1a99249: 1 profili


In [8]:
# Eksperyment ML: prosta klasteryzacja KMeans na embeddingach

from sklearn.cluster import KMeans

# Weźmy embeddingi dla opisów z JSON-a (bez czytania z bazy)
texts = [rec["description"] for rec in records]
embeddings = generate_embeddings_batch(texts)

X = np.array(embeddings)
print("Macierz embeddingów:", X.shape)

# Przy niewielkiej liczbie rekordów spróbujmy np. 3 klastry
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(X)

# Podgląd wyników w DataFrame
ml_df = pd.DataFrame(
    {
        "email": [rec["email"] for rec in records],
        "cluster": cluster_labels,
    }
)

display(ml_df.head())
print("\nLiczność klastrów:")
print(ml_df["cluster"].value_counts())

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.48it/s]

Macierz embeddingów: (27, 384)


,email,cluster
0,test0004@zebra.pl,0
1,test0005@zebra.pl,2
2,test0006@zebra.pl,0
3,test0007@zebra.pl,1
4,test0008@zebra.pl,0



Liczność klastrów:
cluster
0    14
1     9
2     4
Name: count, dtype: int64


In [9]:
def run_full_seed(dry_run: bool = False) -> None:
    """Wysokopoziomowa funkcja do ponownego uruchamiania całego seeda.

    Kroki:
    - wczytanie JSON,
    - stworzenie użytkowników,
    - stworzenie / aktualizacja profili objawów i dopasowanie grup.
    """
    global DRY_RUN, records
    DRY_RUN = dry_run

    # wczytaj dane
    records = load_dataset(DATA_PATH)
    logger.info("Start seeda. Rekordów: %d (DRY_RUN=%s)", len(records), DRY_RUN)

    created = 0
    existing = 0
    profiles_created = 0
    profiles_updated = 0

    with get_db() as db:
        for rec in records:
            # użytkownik
            user_before = db.query(User).filter(User.email == rec["email"]).first()
            user = create_user_from_record(db, rec)
            if user_before is None:
                created += 1
            else:
                existing += 1

            # profil objawów
            existing_profile = (
                db.query(SymptomProfile)
                .filter(SymptomProfile.user_id == user.id)
                .first()
            )
            profile = create_or_update_symptom_profile(db, user, rec["description"])
            if existing_profile is None:
                profiles_created += 1
            else:
                profiles_updated += 1

        if not DRY_RUN:
            # Uzupełnienie charakterystyk grup (gdy Celery nie działa).
            group_ids = {row[0] for row in db.query(SymptomProfile.group_id).all() if row[0] is not None}
            for gid in group_ids:
                update_group_characteristics(db, str(gid))

        if DRY_RUN:
            logger.info("DRY_RUN=True — wykonuję rollback zmian")
            db.rollback()
        else:
            logger.info("Zatwierdzam zmiany w bazie")

    print("Seed zakończony:\n")
    print(f"  Nowo utworzonych użytkowników: {created}")
    print(f"  Użytkownicy już istniejący:   {existing}")
    print(f"  Nowo utworzonych profili:     {profiles_created}")
    print(f"  Zaktualizowanych profili:     {profiles_updated}")


# Przykład użycia:
# run_full_seed(dry_run=True)  # test bez zapisywania do bazy
# run_full_seed(dry_run=False) # realny seed do bazy